# 5. AI/BI Dashboard (deliverable 3.5)

Builds the **Canada Airbnb** AI/BI dashboard (3 pages: *City overview*, *Cross-city comparison*, *Model*) straight from the gold/silver tables and publishes it via the Lakeview API.

- Page titles are **findings computed from the data** at run time (Lecture 7), so re-running after the city C (Montreal) run updates them automatically.
- Every dataset groups by `city`, so Montreal appears on all pages as soon as it's in `silver_listings` / `gold_model_predictions` — no code change.
- Re-running updates the same dashboard (matched by name) instead of creating a copy.

**Prerequisites:** `02_etl_pipeline` run for each city, then `03_price_model`.

In [0]:
dbutils.widgets.text("catalog", spark.catalog.currentCatalog())
dbutils.widgets.text("schema", "airbnb_lakehouse")
dbutils.widgets.text("dashboard_name", "Canada Airbnb — Market & Price Model")
dbutils.widgets.text("parent_path", "/Shared/canada-airbnb/dashboards")
dbutils.widgets.text("training_cities", "toronto,vancouver")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
S = f"{CATALOG}.{SCHEMA}"
DASH_NAME = dbutils.widgets.get("dashboard_name").strip()
PARENT = dbutils.widgets.get("parent_path").strip()
TRAIN_CITIES = [c.strip().lower() for c in dbutils.widgets.get("training_cities").split(",")]

for t in ["silver_listings", "gold_listings_features", "gold_model_predictions"]:
    assert spark.catalog.tableExists(f"{S}.{t}"), f"{S}.{t} not found — run 02_etl_pipeline (per city) and 03_price_model first"
print("Using schema:", S)

## Dashboard definition (datasets + widgets)

In [0]:
import json, re

# ----------------------------------------------------------------------------------
# Lakeview (AI/BI) dashboard builder. `S` = fully-qualified schema, e.g. workspace.airbnb_lakehouse
# `titles` = dict of finding-titles per page; `pred_filter` = extra WHERE for test-only rows.
# ----------------------------------------------------------------------------------

def datasets(S, pred_filter=""):
    L = f"""SELECT city, id, host_id, room_type, neighbourhood_cleansed AS neighbourhood,
  accommodates, price_clean AS price, availability_365,
  try_cast(review_scores_rating AS DOUBLE) AS review_score,
  try_cast(number_of_reviews AS INT) AS number_of_reviews
FROM {S}.silver_listings"""
    P = f"""SELECT p.city, p.id, p.room_type, p.accommodates, f.neighbourhood_cleansed AS neighbourhood,
  p.actual_price, p.predicted_price, p.baseline_price,
  p.predicted_price - p.actual_price AS error,
  abs(p.predicted_price - p.actual_price) AS abs_error,
  abs(p.baseline_price - p.actual_price) AS baseline_abs_error
FROM {S}.gold_model_predictions p
LEFT JOIN {S}.gold_listings_features f ON p.id = f.id AND p.city = f.city
{pred_filter}"""
    q = {
        "ds_city_kpis": ("City KPIs", f"""WITH l AS ({L})
SELECT initcap(city) AS city,
  count(*) AS listings,
  count(DISTINCT host_id) AS hosts,
  round(count(*) / count(DISTINCT host_id), 2) AS listings_per_host,
  percentile_approx(price, 0.5) AS median_price,
  round(avg(price), 1) AS mean_price,
  percentile_approx(availability_365, 0.5) AS median_availability_365,
  round(avg(review_score), 2) AS avg_review_score,
  round(100 * avg(CASE WHEN room_type = 'entire home/apt' THEN 1 ELSE 0 END), 1) AS pct_entire_home
FROM l GROUP BY city ORDER BY listings DESC"""),
        "ds_room_mix": ("Room type mix", f"""WITH l AS ({L})
SELECT initcap(city) AS city, room_type,
  count(*) AS listings,
  round(100 * count(*) / sum(count(*)) OVER (PARTITION BY city), 1) AS pct_of_city,
  percentile_approx(price, 0.5) AS median_price
FROM l GROUP BY city, room_type"""),
        "ds_neigh": ("Neighbourhoods by price", f"""WITH l AS ({L}), n AS (
  SELECT initcap(city) AS city, neighbourhood, count(*) AS listings,
    percentile_approx(price, 0.5) AS median_price
  FROM l GROUP BY city, neighbourhood HAVING count(*) >= 30)
SELECT city, neighbourhood, listings, median_price, neighbourhood_label FROM (
  SELECT *, concat(neighbourhood, ' (', city, ')') AS neighbourhood_label,
    row_number() OVER (PARTITION BY city ORDER BY median_price DESC) AS rn FROM n) WHERE rn <= 8"""),
        "ds_neigh_count": ("Neighbourhoods by listings", f"""WITH l AS ({L})
SELECT city, neighbourhood, neighbourhood_label, listings, median_price FROM (
  SELECT initcap(city) AS city, neighbourhood, concat(neighbourhood, ' (', initcap(city), ')') AS neighbourhood_label,
    count(*) AS listings, percentile_approx(price, 0.5) AS median_price,
    row_number() OVER (PARTITION BY city ORDER BY count(*) DESC) AS rn
  FROM l GROUP BY city, neighbourhood) WHERE rn <= 8"""),
        "ds_avail": ("Availability distribution", f"""WITH l AS ({L})
SELECT initcap(city) AS city,
  CASE WHEN availability_365 = 0 THEN '1) 0 days'
       WHEN availability_365 <= 90 THEN '2) 1-90'
       WHEN availability_365 <= 180 THEN '3) 91-180'
       WHEN availability_365 <= 270 THEN '4) 181-270'
       ELSE '5) 271-365' END AS availability_bucket,
  round(100 * count(*) / sum(count(*)) OVER (PARTITION BY initcap(city)), 1) AS pct_listings
FROM l GROUP BY 1, 2"""),
        "ds_reviews": ("Review score distribution", f"""WITH l AS ({L})
SELECT initcap(city) AS city,
  CASE WHEN review_score IS NULL THEN '0) no reviews'
       WHEN review_score < 4.5 THEN '1) < 4.5'
       WHEN review_score < 4.75 THEN '2) 4.50-4.74'
       WHEN review_score < 4.9 THEN '3) 4.75-4.89'
       ELSE '4) 4.90-5.00' END AS review_bucket,
  round(100 * count(*) / sum(count(*)) OVER (PARTITION BY initcap(city)), 1) AS pct_listings
FROM l GROUP BY 1, 2"""),
        "ds_price_dist": ("Price distribution", f"""WITH l AS ({L})
SELECT initcap(city) AS city,
  CASE WHEN price < 75 THEN '1) < 75'
       WHEN price < 125 THEN '2) 75-124'
       WHEN price < 200 THEN '3) 125-199'
       WHEN price < 300 THEN '4) 200-299'
       WHEN price < 500 THEN '5) 300-499'
       ELSE '6) 500+' END AS price_bucket,
  round(100 * count(*) / sum(count(*)) OVER (PARTITION BY initcap(city)), 1) AS pct_listings
FROM l GROUP BY 1, 2"""),
        "ds_price_by_guests": ("Median price by guests", f"""WITH l AS ({L})
SELECT initcap(city) AS city, accommodates, count(*) AS listings,
  percentile_approx(price, 0.5) AS median_price
FROM l WHERE accommodates BETWEEN 1 AND 10 GROUP BY 1, 2"""),
        "ds_host_conc": ("Host concentration", f"""WITH l AS ({L}), h AS (
  SELECT city, host_id, count(*) AS n FROM l GROUP BY city, host_id)
SELECT initcap(city) AS city,
  round(100 * sum(CASE WHEN n >= 2 THEN n ELSE 0 END) / sum(n), 1) AS pct_listings_multi_host,
  round(100 * sum(CASE WHEN n >= 10 THEN n ELSE 0 END) / sum(n), 1) AS pct_listings_10plus_host
FROM h GROUP BY city"""),
        "ds_mae": ("MAE by city", f"""WITH p AS ({P})
SELECT initcap(city) AS city, count(*) AS listings,
  round(avg(abs_error), 2) AS model_mae, round(avg(baseline_abs_error), 2) AS baseline_mae,
  round(100 * (1 - avg(abs_error) / avg(baseline_abs_error)), 1) AS pct_better_than_baseline,
  percentile_approx(actual_price, 0.5) AS median_actual_price,
  round(100 * avg(abs_error) / percentile_approx(actual_price, 0.5), 1) AS mae_pct_of_median_price
FROM p GROUP BY city
UNION ALL
SELECT 'All cities', count(*), round(avg(abs_error), 2), round(avg(baseline_abs_error), 2),
  round(100 * (1 - avg(abs_error) / avg(baseline_abs_error)), 1),
  percentile_approx(actual_price, 0.5),
  round(100 * avg(abs_error) / percentile_approx(actual_price, 0.5), 1)
FROM p"""),
        "ds_mae_long": ("MAE model vs baseline", f"""WITH p AS ({P})
SELECT initcap(city) AS city, 'Model' AS method, round(avg(abs_error), 2) AS mae FROM p GROUP BY city
UNION ALL
SELECT initcap(city), 'Baseline (median by room type x guests)', round(avg(baseline_abs_error), 2) FROM p GROUP BY city"""),
        "ds_scatter": ("Predicted vs actual (sample)", f"""WITH p AS ({P})
SELECT city, actual_price, predicted_price FROM (
  SELECT initcap(city) AS city, actual_price, round(predicted_price, 0) AS predicted_price,
    row_number() OVER (PARTITION BY city ORDER BY xxhash64(id)) AS rn
  FROM p WHERE actual_price <= 1500) WHERE rn <= 2500"""),
        "ds_err_room": ("Error by room type", f"""WITH p AS ({P})
SELECT initcap(city) AS city, room_type, count(*) AS listings,
  round(avg(abs_error), 2) AS mae, round(avg(error), 2) AS mean_signed_error
FROM p GROUP BY 1, 2"""),
        "ds_err_neigh": ("Error by neighbourhood", f"""WITH p AS ({P})
SELECT city, neighbourhood_label, listings, mae, mean_signed_error FROM (
  SELECT initcap(city) AS city, concat(neighbourhood, ' (', initcap(city), ')') AS neighbourhood_label,
    count(*) AS listings, round(avg(abs_error), 2) AS mae, round(avg(error), 2) AS mean_signed_error,
    row_number() OVER (PARTITION BY initcap(city) ORDER BY abs(avg(error)) DESC) AS rn
  FROM p GROUP BY city, neighbourhood HAVING count(*) >= 20) WHERE rn <= 6"""),
        "ds_under": ("Most under-priced listings", f"""WITH p AS ({P})
SELECT city, id, room_type, accommodates, neighbourhood, actual_price, predicted_price, error FROM (
  SELECT initcap(city) AS city, id, room_type, accommodates, neighbourhood,
    actual_price, round(predicted_price, 0) AS predicted_price, round(error, 0) AS error,
    row_number() OVER (PARTITION BY initcap(city) ORDER BY error ASC) AS rn
  FROM p) WHERE rn <= 15"""),
        "ds_over": ("Most over-priced listings", f"""WITH p AS ({P})
SELECT city, id, room_type, accommodates, neighbourhood, actual_price, predicted_price, error FROM (
  SELECT initcap(city) AS city, id, room_type, accommodates, neighbourhood,
    actual_price, round(predicted_price, 0) AS predicted_price, round(error, 0) AS error,
    row_number() OVER (PARTITION BY initcap(city) ORDER BY error DESC) AS rn
  FROM p) WHERE rn <= 15"""),
    }
    return [{"name": k, "displayName": v[0], "queryLines": [ln + "\n" for ln in v[1].split("\n")]}
            for k, v in q.items()]


# ---------------- widget helpers ----------------
_n = [0]
def _id(p):
    _n[0] += 1
    return f"{p}{_n[0]:03d}"

def text(md, x, y, w, h):
    return {"widget": {"name": _id("txt"), "multilineTextboxSpec": {"lines": md.split("\n")}},
            "position": {"x": x, "y": y, "width": w, "height": h}}

def _agg_field(f):
    m = re.match(r"(sum|avg|max|min)\((\w+)\)", f)
    if m:
        return {"name": f, "expression": f"{m.group(1).upper()}(`{m.group(2)}`)"}
    return {"name": f, "expression": f"`{f}`"}

def chart(kind, ds, title, x, y, color=None, xlab=None, ylab=None, pos=(0, 0, 3, 6),
          x_type="categorical", y_type="quantitative", disagg=False, desc=None):
    fields = [x, y] + ([color] if color else [])
    enc = {"x": {"fieldName": x, "scale": {"type": x_type}, "displayName": xlab or x},
           "y": {"fieldName": y, "scale": {"type": y_type}, "displayName": ylab or y}}
    if color:
        enc["color"] = {"fieldName": color, "scale": {"type": "categorical"}, "displayName": "City" if color == "city" else color}
    frame = {"showTitle": True, "title": title}
    if desc:
        frame.update({"showDescription": True, "description": desc})
    return {"widget": {"name": _id("w"),
                       "queries": [{"name": "main_query", "query": {"datasetName": ds,
                                    "fields": [_agg_field(f) if not disagg else {"name": f, "expression": f"`{f}`"} for f in fields],
                                    "disaggregated": disagg}}],
                       "spec": {"version": 3, "widgetType": kind, "encodings": enc, "frame": frame,
                                "data": {"queryName": "main_query"}}},
            "position": dict(zip(["x", "y", "width", "height"], pos))}

def _col(f, i, typ):
    return {"fieldName": f, "booleanValues": ["false", "true"], "imageUrlTemplate": "{{ @ }}",
            "imageTitleTemplate": "{{ @ }}", "imageWidth": "", "imageHeight": "",
            "linkUrlTemplate": "{{ @ }}", "linkTextTemplate": "{{ @ }}", "linkTitleTemplate": "{{ @ }}",
            "linkOpenInNewTab": True, "type": typ, "displayAs": typ, "visible": True, "order": i,
            "title": f, "allowSearch": False, "alignContent": "right" if typ != "string" else "left",
            "allowHTML": False, "highlightLinks": False, "useMonospaceFont": False,
            "preserveWhitespace": False, "displayName": f}

def table(ds, title, cols, pos, desc=None):
    frame = {"showTitle": True, "title": title}
    if desc:
        frame.update({"showDescription": True, "description": desc})
    return {"widget": {"name": _id("tbl"),
                       "queries": [{"name": "main_query", "query": {"datasetName": ds,
                                    "fields": [{"name": c, "expression": f"`{c}`"} for c, _ in cols],
                                    "disaggregated": True}}],
                       "spec": {"version": 1, "widgetType": "table",
                                "encodings": {"columns": [_col(c, i, t) for i, (c, t) in enumerate(cols)]},
                                "invisibleColumns": [], "allowHTMLByDefault": False, "itemsPerPage": 25,
                                "paginationSize": "default", "condensed": True, "withRowNumber": False,
                                "frame": frame, "data": {"queryName": "main_query"}}},
            "position": dict(zip(["x", "y", "width", "height"], pos))}


def build(S, titles, pred_filter="", model_note=""):
    # Lakeview canvas grid is 12 columns wide: full width = 12, half = 6.
    _n[0] = 0
    s, i, f = "string", "integer", "float"
    p1 = [
        text(f"# {titles['overview']}\n{titles.get('overview_sub', '')}", 0, 0, 12, 3),
        table("ds_city_kpis", "Key metrics per city (prices in CAD per night)",
              [("city", s), ("listings", i), ("hosts", i), ("listings_per_host", f), ("median_price", f),
               ("mean_price", f), ("median_availability_365", i), ("avg_review_score", f), ("pct_entire_home", f)],
              (0, 3, 12, 4)),
        chart("bar", "ds_room_mix", "Room-type mix: % of listings", "room_type", "sum(pct_of_city)", "city",
              "Room type", "% of city listings", (0, 7, 6, 7)),
        chart("bar", "ds_room_mix", "Median nightly price by room type (CAD)", "room_type", "avg(median_price)", "city",
              "Room type", "Median price, CAD", (6, 7, 6, 7)),
        chart("bar", "ds_neigh", "Most expensive neighbourhoods (median price, ≥30 listings)", "avg(median_price)",
              "neighbourhood_label", "city", "Median price, CAD", "Neighbourhood", (0, 14, 6, 8),
              x_type="quantitative", y_type="categorical"),
        chart("bar", "ds_neigh_count", "Neighbourhoods with the most listings", "sum(listings)",
              "neighbourhood_label", "city", "Listings", "Neighbourhood", (6, 14, 6, 8),
              x_type="quantitative", y_type="categorical"),
        chart("bar", "ds_avail", "Availability over the next 365 days: % of listings", "availability_bucket",
              "sum(pct_listings)", "city", "Days available", "% of listings", (0, 22, 6, 7)),
        chart("bar", "ds_reviews", "Review scores: % of listings", "review_bucket", "sum(pct_listings)", "city",
              "Review score (rating)", "% of listings", (6, 22, 6, 7)),
    ]
    p2 = [
        text(f"# {titles['compare']}\n{titles.get('compare_sub', '')}", 0, 0, 12, 3),
        table("ds_city_kpis", "Side by side", [("city", s), ("listings", i), ("hosts", i), ("median_price", f),
              ("mean_price", f), ("median_availability_365", i), ("avg_review_score", f), ("pct_entire_home", f)],
              (0, 3, 12, 4)),
        chart("line", "ds_price_by_guests", "Price ladder: median price by number of guests", "accommodates",
              "avg(median_price)", "city", "Guests (accommodates)", "Median price, CAD", (0, 7, 6, 7)),
        chart("bar", "ds_price_dist", "Price distribution: % of listings per price band (CAD)", "price_bucket",
              "sum(pct_listings)", "city", "Nightly price band", "% of listings", (6, 7, 6, 7)),
        chart("bar", "ds_host_conc", "Professional hosts: % of listings run by hosts with 2+ listings", "city",
              "sum(pct_listings_multi_host)", None, "City", "% of listings", (0, 14, 6, 7)),
        chart("bar", "ds_host_conc", "% of listings run by hosts with 10+ listings", "city",
              "sum(pct_listings_10plus_host)", None, "City", "% of listings", (6, 14, 6, 7)),
    ]
    p3 = [
        text(f"# {titles['model']}\n{titles.get('model_sub', '')}" + (f"\n\n{model_note}" if model_note else ""), 0, 0, 12, 4),
        table("ds_mae", "MAE (CAD) — model vs baseline", [("city", s), ("listings", i), ("model_mae", f),
              ("baseline_mae", f), ("pct_better_than_baseline", f), ("median_actual_price", f),
              ("mae_pct_of_median_price", f)], (0, 4, 12, 4)),
        chart("bar", "ds_mae_long", "MAE by city: model vs baseline (CAD)", "city", "avg(mae)", "method",
              "City", "MAE, CAD", (0, 8, 6, 7)),
        chart("scatter", "ds_scatter", "Predicted vs actual price (sample, ≤ 1,500 CAD)", "actual_price",
              "predicted_price", "city", "Actual price, CAD", "Predicted price, CAD", (6, 8, 6, 7),
              x_type="quantitative", disagg=True),
        chart("bar", "ds_err_room", "MAE by room type (CAD)", "room_type", "avg(mae)", "city",
              "Room type", "MAE, CAD", (0, 15, 6, 7)),
        chart("bar", "ds_err_neigh", "Neighbourhoods with the largest systematic error (mean predicted − actual)",
              "avg(mean_signed_error)", "neighbourhood_label", "city", "Mean signed error, CAD (<0 = under-priced)",
              "Neighbourhood", (6, 15, 6, 8), x_type="quantitative", y_type="categorical"),
        table("ds_under", "Most under-priced listings (model far below actual)",
              [("city", s), ("id", s), ("room_type", s), ("accommodates", i), ("neighbourhood", s),
               ("actual_price", f), ("predicted_price", f), ("error", f)], (0, 23, 6, 8)),
        table("ds_over", "Most over-priced listings (model far above actual)",
              [("city", s), ("id", s), ("room_type", s), ("accommodates", i), ("neighbourhood", s),
               ("actual_price", f), ("predicted_price", f), ("error", f)], (6, 23, 6, 8)),
    ]
    pages = [("p_overview", "City overview", p1), ("p_compare", "Cross-city comparison", p2), ("p_model", "Model", p3)]
    return {"datasets": datasets(S, pred_filter),
            "pages": [{"name": n, "displayName": d, "layout": l, "pageType": "PAGE_TYPE_CANVAS"} for n, d, l in pages],
            "uiSettings": {"theme": {"widgetHeaderAlignment": "ALIGNMENT_UNSPECIFIED"}}}


## Test-set only?
`gold_model_predictions` holds predictions for **every** listing (train + test). Errors on training rows are optimistic, so if a `split` column exists the Model page uses only `split = 'test'` rows. If it doesn't, the page shows a warning plus the official test-set MAE from `03_price_model`.

In [0]:
pred_cols = [c.lower() for c in spark.table(f"{S}.gold_model_predictions").columns]
HAS_SPLIT = "split" in pred_cols
PRED_FILTER = "WHERE p.split = 'test'" if HAS_SPLIT else ""
MODEL_NOTE = ("Errors below are on the **held-out test set** (20% per city)." if HAS_SPLIT else
  "⚠️ Errors below are computed on **all listings (train + test)** because `gold_model_predictions` has no `split` column, so they are optimistic. "
  "Official test-set MAE from `03_price_model`: Toronto **61.24**, Vancouver **92.25**, both **68.53 CAD** vs baseline **109.32 CAD**.")
print("split column present:", HAS_SPLIT)

## Findings → page titles

In [0]:
import pandas as pd
q = lambda name: spark.sql("".join(next(d for d in datasets(S, PRED_FILTER) if d["name"] == name)["queryLines"])).toPandas()

kpi = q("ds_city_kpis").sort_values("listings", ascending=False).reset_index(drop=True)
room = q("ds_room_mix")
hosts = q("ds_host_conc")
mae = q("ds_mae")
display(kpi); display(mae)

def fmt(x): return f"{x:,.0f}"

# --- Page 1: overview ---
big, small = kpi.iloc[0], kpi.iloc[-1]
by_price = kpi.sort_values("median_price", ascending=False)
hi, lo = by_price.iloc[0], by_price.iloc[-1]
t_overview = (f"{big.city} is the largest market ({fmt(big.listings)} listings, {big.listings/small.listings:.1f}× {small.city}), "
              f"{'and also' if hi.city == big.city else 'but'} {hi.city} is the priciest — median {fmt(hi.median_price)} vs {fmt(lo.median_price)} CAD a night")
sub_overview = " · ".join(f"**{r.city}**: {r.pct_entire_home:.0f}% entire homes, median availability {r.median_availability_365:.0f} days/yr, avg rating {r.avg_review_score:.2f}"
                          for r in kpi.itertuples())

# --- Page 2: comparison ---
piv = room[room.listings >= 50].pivot(index="room_type", columns="city", values="median_price").dropna()
gap_txt = ""
if hi.city in piv and lo.city in piv and len(piv):
    g = (piv[hi.city] / piv[lo.city] - 1) * 100
    gap_txt = f"; the gap is widest for {g.idxmax()} (+{g.max():.0f}%)"
h = hosts.set_index("city")
prof = h.pct_listings_multi_host.idxmax()
t_compare = f"{hi.city} charges {(hi.median_price/lo.median_price-1)*100:.0f}% more than {lo.city} at the median{gap_txt}"
sub_compare = (f"Professional hosts (2+ listings) run {h.loc[prof].pct_listings_multi_host:.0f}% of listings in {prof}"
               + "".join(f", {v:.0f}% in {c}" for c, v in h.pct_listings_multi_host.items() if c != prof) + ".")

# --- Page 3: model ---
m = mae[mae.city != "All cities"].set_index("city")
allrow = mae[mae.city == "All cities"].iloc[0]
train = m[m.index.str.lower().isin(TRAIN_CITIES)]
new = m[~m.index.str.lower().isin(TRAIN_CITIES)]
worst, best = train.model_mae.idxmax(), train.model_mae.idxmin()
pb = allrow.pct_better_than_baseline
t_model = ((f"The model beats the median baseline by {pb:.0f}%, " if pb > 0 else f"The model is {-pb:.0f}% WORSE than the median baseline, ") +
           f"but its error in {worst} is {train.loc[worst].model_mae/train.loc[best].model_mae:.1f}× {best}'s")
if len(new):
    c = new.index[0]
    t_model += (f" — and on unseen {c} MAE jumps to {new.loc[c].model_mae:.0f} CAD "
                f"(baseline {new.loc[c].baseline_mae:.0f}) vs {train.model_mae.mean():.0f} on training cities")
sub_model = " · ".join(f"**{c}**: MAE {r.model_mae:.0f} CAD vs baseline {r.baseline_mae:.0f} ({r.mae_pct_of_median_price:.0f}% of median price)"
                       for c, r in m.iterrows())

TITLES = {"overview": t_overview, "overview_sub": sub_overview,
          "compare": t_compare, "compare_sub": sub_compare,
          "model": t_model, "model_sub": sub_model}
for k, v in TITLES.items(): print(f"{k:13s} {v}")

## Create / update and publish the dashboard

In [0]:
import json
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.dashboards import Dashboard

w = WorkspaceClient()
spec = json.dumps(build(S, TITLES, PRED_FILTER, MODEL_NOTE))

warehouses = list(w.warehouses.list())
assert warehouses, "No SQL warehouse found — start the Serverless Starter Warehouse"
WH = next((x for x in warehouses if "starter" in (x.name or "").lower()), warehouses[0]).id

try:
    w.workspace.mkdirs(PARENT)
except Exception as e:
    print("mkdirs:", e)

existing = next((d for d in w.lakeview.list() if d.display_name == DASH_NAME), None)
dash = Dashboard(display_name=DASH_NAME, serialized_dashboard=spec, warehouse_id=WH,
                 parent_path=None if existing else PARENT)
if existing:
    d = w.lakeview.update(existing.dashboard_id, dashboard=dash)
    print("Updated dashboard", d.dashboard_id)
else:
    d = w.lakeview.create(dashboard=dash)
    print("Created dashboard", d.dashboard_id)

w.lakeview.publish(d.dashboard_id, warehouse_id=WH, embed_credentials=True)
url = f"{w.config.host.rstrip('/')}/dashboardsv3/{d.dashboard_id}"
displayHTML(f'<a href="{url}" target="_blank">Open dashboard: {DASH_NAME}</a>')

### Optional: save the definition for the repo
Downloads-friendly copy of the dashboard JSON (import via **Dashboards → Create → Import dashboard from file**).

In [0]:
out = f"/Workspace{PARENT}/airbnb_dashboard_definition.json"
with open(out, "w") as f:
    f.write(json.dumps(json.loads(spec), indent=2))
print("Saved", out)